# Similarity-aware Label Smoothing

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
from hyperparams import dataset, batch_size, device
import pandas as pd
from confidence import histogram_ece, top_label_ece

ImportError: cannot import name 'top_label_ece' from 'confidence' (/content/confidence.py)

## Hyperparams

In [2]:
dataset = "cifar100"
lr = 5e-4
epochs = 20
num_classes = int(dataset.removeprefix("cifar"))

## Models

In [3]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2,2)
        self.fc1   = nn.Linear(64*7*7, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


### CIFAR-100

In [4]:
def RESNET18():
    model = models.resnet18(weights = None)

    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model

## Training Utils

### Accuracy Measure

In [5]:
def accuracy(model, loader, k = (1, 5)):
    model.eval()
    correct = {key:0 for key in k}
    total = 0

    maxk = max(k)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)

            _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)

            y_expand = y.view(-1, 1).expand_as(pred)

            for key in k:
                correct[key] += (pred[:, :key] == y.view(-1, 1)).any(dim=1).sum().item()
            total += y.size(0)

    return {key: correct[key] / total for key in k}

### Label Smoothing

In [6]:
path = f"scores/{dataset}_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)
print(dists)
def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

def scores(y, k=5, alpha=5.0):
    class_dists = dists[y]

    mask = torch.eye(class_dists.shape[1], device=device)[y]
    class_dists = class_dists.masked_fill(mask.bool(), float('inf'))
    sims = F.softmax(torch.exp(-class_dists), dim=1)
    sims.scatter_(1, y.unsqueeze(1), 0.0)

    sims = sims ** alpha

    topk_values, topk_indices = torch.topk(sims, k, dim=1)
    result = torch.zeros_like(sims)
    result.scatter_(1, topk_indices, topk_values)

    result = result / (result.sum(dim=1, keepdim=True) + 1e-9)

    return result

def similarity_aware_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return (1 - epsilon) * y_onehot + epsilon * scores(y)


tensor([[0.0000, 2.2437, 1.8875,  ..., 2.4016, 1.5960, 2.2348],
        [2.2437, 0.0000, 1.7106,  ..., 1.3664, 1.6650, 1.6519],
        [1.8875, 1.7106, 0.0000,  ..., 1.1100, 1.0685, 1.0712],
        ...,
        [2.4016, 1.3664, 1.1100,  ..., 0.0000, 1.3440, 0.9323],
        [1.5960, 1.6650, 1.0685,  ..., 1.3440, 0.0000, 1.3156],
        [2.2348, 1.6519, 1.0712,  ..., 0.9323, 1.3156, 0.0000]],
       device='cuda:0')


## Training Loop

In [7]:
def train(model, train_loader, test_loader, optimizer=None, scheduler=None, num_classes=100, epochs=10, epsilon=0.1):
    classwise_target = similarity_aware_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)

    print(classwise_target)
    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device), y.to(device)

            targets = classwise_target[y]

            optimizer.zero_grad()
            logits = model(x)
            loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        test_acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")

## RUN Experiments

In [ ]:
train_loader, test_loader = get_data_loaders(dataset="cifar100")
print(train_loader.dataset.classes)
model = RESNET18().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.2, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_classes=num_classes,
    epochs=10,
    epsilon=0.1,
)

['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine', 'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout', 'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree',

Epoch 1/10 | Loss: 3.9804 | Test Acc: 0.1128 | Top-5 Test Acc: 0.3471


In [ ]:
print(train_loader.dataset.classes)

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [ ]:
logits_list = []
labels_list = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        logits_list.append(logits)
        labels_list.append(y)

logits_all = torch.cat(logits_list, dim=0)
labels_all = torch.cat(labels_list, dim=0)

ece = histogram_ece(logits_all, labels_all)
print("ECE:", ece)


ECE: 0.0018238520715385675


In [ ]:
def top_label_ece(logits, labels, n_bins=15):
    probs = F.softmax(logits, dim=1)
    confidences, predictions = probs.max(dim=1)

    correctness = (predictions == labels).float()
    bin_edges = torch.linspace(0, 1.0, n_bins + 1, device=logits.device)

    ece = torch.zeros(1, device=logits.device)

    for m in range(n_bins):
        start = bin_edges[m]
        end = bin_edges[m + 1]

        in_bin = (confidences > start) & (confidences <= end)
        num_in_bin = in_bin.sum()

        if num_in_bin > 0:
            acc_in_bin = correctness[in_bin].mean()
            conf_in_bin = confidences[in_bin].mean()
            ece += (num_in_bin.float() / len(confidences)) * torch.abs(acc_in_bin - conf_in_bin)

    return ece.item()

ece = top_label_ece(logits_all, labels_all)
print("ECE:", ece)

ECE: 0.073938749730587
